In [ ]:
from geopy.geocoders import Nominatim
from geopy.distance import great_circle
from ipyleaflet import Map, Marker, CircleMarker, Popup, LayersControl, basemaps
from ipywidgets import HTML
from IPython.display import display

# -----------------------------------------
# 1. GEOCODING FUNCTION
# -----------------------------------------
def geocode_city(city_name):
    """Convert a city name into (latitude, longitude) coordinates using Nominatim."""
    geolocator = Nominatim(user_agent="geo_analysis")
    location = geolocator.geocode(city_name)
    if location:
        return (location.latitude, location.longitude)
    return None

# -----------------------------------------
# 2. DISTANCE CALCULATION FUNCTION
# -----------------------------------------
def calculate_distance(point1, point2):
    """Calculate great-circle distance in kilometers between two (lat, lon) points."""
    return great_circle(point1, point2).kilometers

# -----------------------------------------
# 3. MAIN PROGRAM
# -----------------------------------------
city_name = input("Enter a city name to find nearby big cities: ")

# Get coordinates of input city
city_coords = geocode_city(city_name)

if city_coords is None:
    print("City not found.")
else:
    print(f"Coordinates of {city_name}: {city_coords}")

    # Dictionary of big cities in Germany with their coordinates
    big_cities = {
        "Berlin": (52.5200, 13.4050),
        "Hamburg": (53.5511, 9.9937),
        "Munich": (48.1351, 11.5820),
        "Cologne": (50.9375, 6.9603),
        "Frankfurt": (50.1109, 8.6821),
        "Stuttgart": (48.7758, 9.1829),
        "Düsseldorf": (51.2277, 6.7735),
        "Leipzig": (51.3397, 12.3731),
        "Dortmund": (51.5136, 7.4653),
        "Essen": (51.4556, 7.0116),
        "Bremen": (53.0793, 8.8017),
        "Dresden": (51.0504, 13.7373),
        "Hanover": (52.3759, 9.7320),
        "Nuremberg": (49.4521, 11.0767),
        "Duisburg": (51.4344, 6.7623)
    }

    # Find nearby cities within 200 km radius
    nearby_cities = {}
    for city, coords in big_cities.items():
        distance = calculate_distance(city_coords, coords)
        if distance <= 200:
            nearby_cities[city] = coords

    # Print nearby cities
    if nearby_cities:
        print("Nearby big cities within 200 km:")
        for city in nearby_cities:
            print(f"- {city}")
    else:
        print("No nearby big cities within 200 km.")

    # -----------------------------------------
    # 4. CREATE INTERACTIVE MAP
    # -----------------------------------------
    m = Map(
        center=(city_coords[0], city_coords[1]),
        zoom=6,
        basemap=basemaps.OpenStreetMap.Mapnik
    )

    # Marker for input city
    input_city_marker = Marker(
        location=(city_coords[0], city_coords[1]),
        title=city_name
    )
    m.add_layer(input_city_marker)

    # Add markers for nearby big cities (orange) with popups
    for city, coords in nearby_cities.items():
        marker = CircleMarker(
            location=(coords[0], coords[1]),
            radius=7,
            color="orange",
            fill_color="orange",
            fill_opacity=0.6
        )
        # Create a popup with the city name
        popup = Popup(
            location=(coords[0], coords[1]),
            child=HTML(f"<b>{city}</b>"),
            close_button=False,
            auto_close=False,
            close_on_escape_key=False
        )
        m.add_layer(marker)
        m.add_layer(popup)

    # Add layer control for toggling layers
    m.add_control(LayersControl())

    # Display the interactive map
    display(m)
